# Q4 - Offline Evaluation Harness

Re-ranking evaluation (AUC, MRR, nDCG@5/@10) over each impression's own
`article_ids_inview`, plus beyond-accuracy metrics (intra-list diversity,
novelty, coverage), cold-start-vs-warm and head-vs-tail slicing, and
bootstrap 95% CIs -- for both BM25 (Q2) and embeddings (Q3), per SPEC.md's
Q4 section. This is a different framing from Q2/Q3's candidate-generation
recall@K: see SPEC.md Q4 #1 for why a `score_inview` re-scoring adapter is
needed instead of just reusing the persisted top-200 lists.

Run top-to-bottom (or via `python evaluation_harness.py`) to rebuild
`data/processed/{dataset}/eval_metrics.json`.

## Setup

In [1]:
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path
import json

import numpy as np
import pandas as pd

from cs4406m26_assignment1c1.bm25 import tokenize, build_index, get_scores
from cs4406m26_assignment1c1.embeddings import mean_pool, cosine_similarity_subset
from cs4406m26_assignment1c1.evaluation import (
    auc_impression,
    mrr,
    ndcg_at_k,
    bootstrap_ci,
    intra_list_diversity,
    novelty,
    coverage,
)


def find_repo_root(marker: str = "pyproject.toml") -> Path:
    for parent in [Path.cwd(), *Path.cwd().parents]:
        if (parent / marker).exists():
            return parent
    raise FileNotFoundError(f"could not locate {marker} above {Path.cwd()}")


ROOT = find_repo_root()
DATA_DIR = ROOT / "data" / "processed"
DATASETS = ["ebnerd", "mind"]
METHODS = ["bm25", "embedding"]
SPLITS = ["val", "test"]

RECENT_N_CLICKS = 20
NDCG_K_VALUES = [5, 10]
TOP_LIST_K = 10  # matches nDCG@10's window, per SPEC.md Q4 #3
HEAD_FRACTION = 0.2
BOOTSTRAP_ITERATIONS = 1000
BOOTSTRAP_SEED = 0

pd.set_option("display.max_columns", 50)
pd.set_option("display.max_colwidth", 80)
pd.set_option("display.width", 160)

feature_store = {
    name: {
        "articles": pd.read_parquet(DATA_DIR / name / "articles.parquet"),
        "behaviors": pd.read_parquet(DATA_DIR / name / "behaviors.parquet"),
        "history": pd.read_parquet(DATA_DIR / name / "history.parquet"),
    }
    for name in DATASETS
}

embedding_paths = {name: DATA_DIR / name / "article_embeddings.parquet" for name in DATASETS}
missing = [str(p) for p in embedding_paths.values() if not p.exists()]
if missing:
    raise FileNotFoundError(
        f"missing article_embeddings.parquet: {missing}. "
        "Run src/compute_embeddings_kaggle.ipynb on Kaggle first (see README.md)."
    )
embeddings_raw = {name: pd.read_parquet(embedding_paths[name]) for name in DATASETS}

# Rebuild BM25 indexes locally (cheap, seconds -- SPEC.md Q2 #1) rather than
# depending on bm25_retrieval.ipynb's in-memory state, which a separate
# notebook process can't see.
bm25_index = {}
title_by_id = {}
for name in DATASETS:
    articles = feature_store[name]["articles"]
    text = articles["title"].fillna("") + " " + articles["abstract"].fillna("")
    doc_tokens = text.map(tokenize).tolist()
    bm25_index[name] = build_index(articles["article_id"].tolist(), doc_tokens)
    title_by_id[name] = dict(zip(articles["article_id"], articles["title"]))

# Corpus embedding matrices, reindexed to articles.parquet order (same pattern as Q3).
corpus = {}
for name in DATASETS:
    articles = feature_store[name]["articles"]
    emb_lookup = dict(zip(embeddings_raw[name]["article_id"], embeddings_raw[name]["embedding"].apply(np.asarray)))
    doc_ids = articles["article_id"].to_numpy()
    missing_ids = set(doc_ids) - set(emb_lookup)
    if missing_ids:
        raise ValueError(f"{name}: {len(missing_ids)} articles have no embedding")
    matrix = np.stack([emb_lookup[aid] for aid in doc_ids]).astype(np.float32)
    corpus[name] = {"doc_ids": doc_ids, "matrix": matrix, "embedding_lookup": emb_lookup}

{name: {"bm25_docs": bm25_index[name].n_docs, "embedding_docs": len(corpus[name]["doc_ids"])} for name in DATASETS}

{'ebnerd': {'bm25_docs': 11777, 'embedding_docs': 11777},
 'mind': {'bm25_docs': 65238, 'embedding_docs': 65238}}

In [2]:
def test_setup_aligned():
    for name in DATASETS:
        articles = feature_store[name]["articles"]
        assert bm25_index[name].n_docs == len(articles)
        assert corpus[name]["matrix"].shape[0] == len(articles)
        assert (corpus[name]["doc_ids"] == articles["article_id"].to_numpy()).all()
        assert not np.isnan(corpus[name]["matrix"]).any()


test_setup_aligned()
print("ok: BM25 indexes and embedding matrices rebuilt/loaded and aligned with articles.parquet")

ok: BM25 indexes and embedding matrices rebuilt/loaded and aligned with articles.parquet


## Metric-formula smoke tests

Hand-computed toy rankings with independently-derived expected values (SPEC.md
Q4 #2), before these functions are trusted against the real feature store.

In [3]:
def test_metric_formulas():
    # AUC: perfect ranking -> 1.0, inverted -> 0.0, all-tied -> 0.5 (this is
    # exactly what a cold-start user's all-zero BM25 score vector produces --
    # see the adapters below, so this case matters, not just a formula edge case)
    assert auc_impression([0.9, 0.1], [1, 0]) == 1.0
    assert auc_impression([0.1, 0.9], [1, 0]) == 0.0
    assert auc_impression([0.5, 0.5, 0.5], [1, 0, 1]) == 0.5

    # MRR: reciprocal rank of the best-scored clicked item
    assert mrr([0.9, 0.1, 0.05], [0, 1, 0]) == 0.5
    assert mrr([0.9, 0.1], [1, 0]) == 1.0

    # nDCG@5: hand-computed DCG/IDCG for a known 3-item ranking with 2 positives
    scores, labels = [0.9, 0.5, 0.1], [0, 1, 1]
    dcg = 0 / np.log2(2) + 1 / np.log2(3) + 1 / np.log2(4)
    idcg = 1 / np.log2(2) + 1 / np.log2(3)
    assert abs(ndcg_at_k(scores, labels, 5) - dcg / idcg) < 1e-9
    assert ndcg_at_k([0.9, 0.1], [1, 0], 5) == 1.0

    # bootstrap CI: point estimate is the plain mean; CI collapses to the point
    # for a constant array; CI always brackets the point for non-constant input
    point, lo, hi = bootstrap_ci([1.0, 1.0, 1.0], seed=0)
    assert point == lo == hi == 1.0
    point, lo, hi = bootstrap_ci([0.0, 1.0], n_iterations=2000, seed=0)
    assert lo <= point <= hi

    # intra-list diversity: 0 for <2 items or an all-same-category list, 1 for all-different
    cat = {"a": "x", "b": "x", "c": "y"}
    assert intra_list_diversity(["a"], cat) == 0.0
    assert intra_list_diversity(["a", "b"], cat) == 0.0
    assert intra_list_diversity(["a", "c"], cat) == 1.0

    # novelty: mean of precomputed per-article scores
    assert novelty(["a", "b"], {"a": 2.0, "b": 4.0}) == 3.0
    assert novelty([], {"a": 2.0}) == 0.0

    # coverage: union of retrieved sets over the catalog size
    assert coverage([["a", "b"], ["b", "c"]], 4) == 0.75


test_metric_formulas()
print("ok: AUC/MRR/nDCG/bootstrap_ci/intra_list_diversity/novelty/coverage match hand-computed values")

ok: AUC/MRR/nDCG/bootstrap_ci/intra_list_diversity/novelty/coverage match hand-computed values


## Cold-start user set (cross-checked against Q2)

Same definition Q2/Q3 use (empty `article_id_sequence`), recomputed here
(separate notebook process) but cross-checked against Q2's already-persisted
`bm25_metrics.json` cold-start impression counts, which must match exactly.
Unlike Q2/Q3's recall@K, cold-start users are **not excluded** from ranking
metrics below -- see the adapters section for why.

In [4]:
def build_coldstart_users(dataset: str) -> set:
    history = feature_store[dataset]["history"]
    behaviors = feature_store[dataset]["behaviors"]
    eval_user_ids = set(behaviors.loc[behaviors["split"].isin(SPLITS), "user_id"])
    history_eval = history[history["user_id"].isin(eval_user_ids)]
    return set(history_eval.loc[history_eval["article_id_sequence"].apply(len) == 0, "user_id"])


coldstart_users = {name: build_coldstart_users(name) for name in DATASETS}
{name: len(coldstart_users[name]) for name in DATASETS}

{'ebnerd': 0, 'mind': 1770}

In [5]:
def test_coldstart_users():
    for name in DATASETS:
        behaviors = feature_store[name]["behaviors"]
        bm25_metrics_path = DATA_DIR / name / "bm25_metrics.json"
        if not bm25_metrics_path.exists():
            continue
        bm25_metrics = json.loads(bm25_metrics_path.read_text())
        for split in SPLITS:
            expected = bm25_metrics["n_impressions"][split]["excluded_coldstart"]
            actual = behaviors[
                (behaviors["split"] == split) & (behaviors["user_id"].isin(coldstart_users[name]))
            ].shape[0]
            assert actual == expected, f"{name}/{split}: cold-start impression count diverged from Q2's bm25_metrics.json"


test_coldstart_users()
print("ok: cold-start user set matches Q2's already-persisted counts exactly")

ok: cold-start user set matches Q2's already-persisted counts exactly


## Train-split popularity (novelty lookup + head/tail articles)

Shared basis for both #3's novelty metric and #4's optional head-vs-tail
slice (SPEC.md Q4 #3/#4): `pop(item) = clicks_train(item) / total_train_clicks`,
add-one-smoothed for never-clicked items so `novelty = -log2(pop)` never hits
`log2(0)`. Head = top 20% of *ever-clicked* articles by train-click-count;
tail = everything else, including never-clicked articles.

In [6]:
def compute_train_click_counts(dataset: str) -> Counter:
    behaviors = feature_store[dataset]["behaviors"]
    train_clicked = behaviors.loc[behaviors["split"] == "train", "article_ids_clicked"]
    counts = Counter()
    for clicked in train_clicked:
        counts.update(clicked)
    return counts


train_click_counts = {name: compute_train_click_counts(name) for name in DATASETS}
train_total_clicks = {name: sum(train_click_counts[name].values()) for name in DATASETS}

novelty_lookup = {}
for name in DATASETS:
    articles = feature_store[name]["articles"]
    counts = train_click_counts[name]
    total = train_total_clicks[name]
    n_articles = len(articles)
    smoothed_zero_pop = 1.0 / (total + n_articles)
    novelty_lookup[name] = {}
    for aid in articles["article_id"]:
        c = counts.get(aid, 0)
        pop = (c / total) if c > 0 else smoothed_zero_pop
        novelty_lookup[name][aid] = -np.log2(pop)

head_articles = {}
for name in DATASETS:
    ever_clicked = sorted(train_click_counts[name].items(), key=lambda kv: -kv[1])
    n_head = max(1, int(len(ever_clicked) * HEAD_FRACTION))
    head_articles[name] = {aid for aid, _ in ever_clicked[:n_head]}

{
    name: {
        "never_clicked_pct": 100 * (1 - len(train_click_counts[name]) / len(feature_store[name]["articles"])),
        "head_articles_share_of_train_clicks_pct": 100
        * sum(train_click_counts[name][aid] for aid in head_articles[name])
        / train_total_clicks[name],
    }
    for name in DATASETS
}

{'ebnerd': {'never_clicked_pct': 91.76360703065298,
  'head_articles_share_of_train_clicks_pct': 49.21256173702358},
 'mind': {'never_clicked_pct': 90.19283239829548,
  'head_articles_share_of_train_clicks_pct': 90.20203779040624}}

In [7]:
def test_train_popularity():
    for name in DATASETS:
        assert all(v >= 0 for v in novelty_lookup[name].values())
        assert set(novelty_lookup[name]) == set(feature_store[name]["articles"]["article_id"])
        assert head_articles[name].issubset(set(train_click_counts[name]))

    # sanity-check against SPEC.md Q4 #3/#4's already-verified real-data figures
    # (small tolerance -- exact figures depend on the pipeline's random-free but
    # not bit-identical aggregation order)
    mind_never_clicked_pct = 100 * (1 - len(train_click_counts["mind"]) / len(feature_store["mind"]["articles"]))
    ebnerd_never_clicked_pct = 100 * (1 - len(train_click_counts["ebnerd"]) / len(feature_store["ebnerd"]["articles"]))
    assert abs(mind_never_clicked_pct - 90.2) < 1.0
    assert abs(ebnerd_never_clicked_pct - 91.8) < 1.0

    mind_head_share = 100 * sum(train_click_counts["mind"][a] for a in head_articles["mind"]) / train_total_clicks["mind"]
    assert abs(mind_head_share - 90.2) < 1.0


test_train_popularity()
print("ok: novelty lookup covers every article and is non-negative; head/tail figures match SPEC.md's verified numbers")

ok: novelty lookup covers every article and is non-negative; head/tail figures match SPEC.md's verified numbers


## score_inview adapters

Uniform signature (SPEC.md Q4 #1): `score_inview(user_id, article_ids_inview)
-> dict[article_id, float]`, one instance per (dataset, method). BM25
memoizes the *last* user's full-corpus score vector -- the ranking-metrics
loop below processes each split sorted by `user_id`, so consecutive calls for
the same user hit the cache and `get_scores` runs once per user (~5ms, per
Q2's benchmark), not once per impression.

Cold-start users (empty history) get an all-zero score vector from
`get_scores`/an empty dict from `cosine_similarity_subset` -- both produce a
well-defined (if uninformative, all-tied -> AUC 0.5) ranking rather than an
exclusion. That is what makes cold-start-vs-warm a genuine *slice* for Q4,
unlike Q2/Q3's recall@K, where a cold-start user has no query at all and must
be excluded outright.

In [8]:
def build_user_query_tokens(article_id_sequence, title_lookup: dict, recent_n: int = RECENT_N_CLICKS) -> list[str]:
    recent_ids = list(article_id_sequence)[-recent_n:]
    titles = [title_lookup.get(aid, "") for aid in recent_ids]
    return tokenize(" ".join(t for t in titles if t))


def build_user_query_vector(article_id_sequence, embedding_lookup: dict, recent_n: int = RECENT_N_CLICKS):
    recent_ids = list(article_id_sequence)[-recent_n:]
    return mean_pool(recent_ids, embedding_lookup)


def make_score_inview_adapters(dataset: str) -> dict:
    id_to_idx = {aid: i for i, aid in enumerate(bm25_index[dataset].doc_ids)}
    history_lookup = feature_store[dataset]["history"].set_index("user_id")["article_id_sequence"]
    title_lookup = title_by_id[dataset]
    embedding_lookup = corpus[dataset]["embedding_lookup"]
    emb_matrix = corpus[dataset]["matrix"]
    emb_doc_ids = corpus[dataset]["doc_ids"]

    bm25_cache = {"user_id": None, "scores": None}

    def bm25_fn(user_id, article_ids_inview):
        if bm25_cache["user_id"] != user_id:
            seq = history_lookup.get(user_id, [])
            query_tokens = build_user_query_tokens(seq, title_lookup)
            bm25_cache["user_id"] = user_id
            bm25_cache["scores"] = get_scores(bm25_index[dataset], query_tokens)
        scores = bm25_cache["scores"]
        return {aid: float(scores[id_to_idx[aid]]) for aid in article_ids_inview}

    def embedding_fn(user_id, article_ids_inview):
        seq = history_lookup.get(user_id, [])
        query_vector = build_user_query_vector(seq, embedding_lookup)
        scored = cosine_similarity_subset(query_vector, emb_matrix, emb_doc_ids, article_ids_inview)
        return {aid: scored.get(aid, 0.0) for aid in article_ids_inview}

    return {"bm25": bm25_fn, "embedding": embedding_fn}


score_inview_adapters = {name: make_score_inview_adapters(name) for name in DATASETS}

In [9]:
def test_score_inview_adapters():
    for name in DATASETS:
        behaviors = feature_store[name]["behaviors"]
        sample = behaviors.iloc[0]
        inview = list(sample["article_ids_inview"])

        for method in METHODS:
            fn = score_inview_adapters[name][method]
            scored = fn(sample["user_id"], inview)
            assert set(scored) == set(inview)
            assert all(np.isfinite(v) for v in scored.values())

        # cold-start user (empty history) -> all-zero (tied) scores, not a crash
        history = feature_store[name]["history"]
        coldstart_rows = history[history["article_id_sequence"].apply(len) == 0]
        if len(coldstart_rows) > 0:
            coldstart_user = coldstart_rows.iloc[0]["user_id"]
            bm25_scored = score_inview_adapters[name]["bm25"](coldstart_user, inview)
            assert set(bm25_scored.values()) == {0.0}
            emb_scored = score_inview_adapters[name]["embedding"](coldstart_user, inview)
            assert set(emb_scored.values()) == {0.0}

        # BM25 cache correctness: repeated calls for the same user return identical scores
        fn = score_inview_adapters[name]["bm25"]
        assert fn(sample["user_id"], inview) == fn(sample["user_id"], inview)


test_score_inview_adapters()
print("ok: score_inview adapters have a uniform signature, no NaN/inf, and score cold-start users as an all-zero tie")

ok: score_inview adapters have a uniform signature, no NaN/inf, and score cold-start users as an all-zero tie


## Per-impression ranking metrics

AUC/MRR/nDCG@5/@10 over each impression's own `article_ids_inview`, scored
by `score_inview` -- the re-ranking framing (SPEC.md Q4 #1), not Q2/Q3's
full-catalog candidate generation. `article_ids_inview` never has a
zero-click or single-candidate impression (verified: min inview size is 5
for EB-NeRD / 2 for MIND, and every impression has >=1 click), so these
metrics are always defined -- no impressions are excluded here, unlike
Q2/Q3's recall@K. Each split is processed sorted by `user_id` so the BM25
adapter's cache is effective. Also records each impression's top-10
re-ranked list (`top10_ids`), reused by the beyond-accuracy metrics next.

In [10]:
def evaluate_ranking(dataset: str, split: str, method: str) -> pd.DataFrame:
    behaviors = feature_store[dataset]["behaviors"]
    split_behaviors = behaviors[behaviors["split"] == split].sort_values("user_id")
    score_fn = score_inview_adapters[dataset][method]
    coldstart = coldstart_users[dataset]
    head = head_articles[dataset]

    records = []
    for user_id, inview, clicked in zip(
        split_behaviors["user_id"], split_behaviors["article_ids_inview"], split_behaviors["article_ids_clicked"]
    ):
        inview_ids = list(inview)
        clicked_set = set(clicked)
        scored = score_fn(user_id, inview_ids)
        scores = np.array([scored[aid] for aid in inview_ids])
        labels = np.array([aid in clicked_set for aid in inview_ids])
        top_order = np.argsort(-scores, kind="stable")[:TOP_LIST_K]

        record = {
            "user_id": user_id,
            "auc": auc_impression(scores, labels),
            "mrr": mrr(scores, labels),
            "is_coldstart": user_id in coldstart,
            "is_head": any(aid in head for aid in clicked_set),
            "top10_ids": [inview_ids[i] for i in top_order],
        }
        for k in NDCG_K_VALUES:
            record[f"ndcg{k}"] = ndcg_at_k(scores, labels, k)
        records.append(record)

    df = pd.DataFrame(records)
    df["dataset"] = dataset
    df["split"] = split
    df["method"] = method
    return df


ranking_results = pd.concat(
    [evaluate_ranking(name, split, method) for name in DATASETS for split in SPLITS for method in METHODS],
    ignore_index=True,
)
ranking_results.shape

(264368, 11)

In [11]:
def test_ranking_metrics():
    expected_total = sum(
        (feature_store[name]["behaviors"]["split"] == split).sum() for name in DATASETS for split in SPLITS
    ) * len(METHODS)
    assert len(ranking_results) == expected_total

    for col in ["auc", "mrr", "ndcg5", "ndcg10"]:
        assert ranking_results[col].between(0.0, 1.0).all()
        assert ranking_results[col].notna().all()

    for name in DATASETS:
        for split in SPLITS:
            n_total = int((feature_store[name]["behaviors"]["split"] == split).sum())
            for method in METHODS:
                subset = ranking_results[
                    (ranking_results["dataset"] == name)
                    & (ranking_results["split"] == split)
                    & (ranking_results["method"] == method)
                ]
                n_evaluated = len(subset)
                n_excluded = 0  # ranking metrics are always defined -- see markdown above
                assert n_evaluated + n_excluded == n_total


test_ranking_metrics()
print("ok: ranking metrics computed for every impression (n_evaluated + n_excluded == n_total), all values in [0,1]")

ok: ranking metrics computed for every impression (n_evaluated + n_excluded == n_total), all values in [0,1]


## Beyond-accuracy metrics (diversity, novelty, coverage)

Intra-list diversity and novelty over each impression's top-10 re-ranked list
(SPEC.md Q4 #3). Coverage is read from Q2/Q3's persisted top-K artifacts, not
the reranked list, since it's a candidate-generation-framed statistic.

In [12]:
category_lookup = {
    name: dict(zip(feature_store[name]["articles"]["article_id"], feature_store[name]["articles"]["category"]))
    for name in DATASETS
}

ranking_results["ild"] = [
    intra_list_diversity(top10, category_lookup[dataset])
    for top10, dataset in zip(ranking_results["top10_ids"], ranking_results["dataset"])
]
ranking_results["novelty"] = [
    novelty(top10, novelty_lookup[dataset]) for top10, dataset in zip(ranking_results["top10_ids"], ranking_results["dataset"])
]


def compute_coverage(dataset: str, method: str) -> float:
    topk_path = DATA_DIR / dataset / f"{method}_topk.parquet"
    topk_df = pd.read_parquet(topk_path)
    n_articles = len(feature_store[dataset]["articles"])
    return coverage(topk_df["retrieved_article_ids"], n_articles)


coverage_metrics = {(name, method): compute_coverage(name, method) for name in DATASETS for method in METHODS}
coverage_metrics

{('ebnerd', 'bm25'): 0.9200984970705612,
 ('ebnerd', 'embedding'): 0.5604992782542243,
 ('mind', 'bm25'): 0.9948036420491125,
 ('mind', 'embedding'): 0.967151046935835}

In [13]:
def test_beyond_accuracy_metrics():
    assert ranking_results["ild"].between(0.0, 1.0).all()
    assert (ranking_results["novelty"] >= 0.0).all()
    for cov in coverage_metrics.values():
        assert 0.0 <= cov <= 1.0

    # identical-category list -> ILD == 0 (already unit-tested on intra_list_diversity
    # directly in the metric-formula smoke tests; confirms the column-building path agrees)
    assert intra_list_diversity(["a", "b"], {"a": "x", "b": "x"}) == 0.0


test_beyond_accuracy_metrics()
print("ok: intra-list diversity in [0,1], novelty >= 0, coverage in [0,1] for both methods")

ok: intra-list diversity in [0,1], novelty >= 0, coverage in [0,1] for both methods


## Slicing

Cold-start vs. warm (required) and head vs. tail (optional), per SPEC.md
Q4 #4. Head/tail is applied at the impression level: an impression counts as
`head` if at least one of its clicked articles is a head item (a single
popular click is enough to signal what the method needs to get right), else
`tail`.

In [14]:
SLICE_DEFINITIONS = {
    "overall": lambda df: pd.Series(True, index=df.index),
    "cold_start": lambda df: df["is_coldstart"],
    "warm": lambda df: ~df["is_coldstart"],
    "head": lambda df: df["is_head"],
    "tail": lambda df: ~df["is_head"],
}
METRIC_COLUMNS = ["auc", "mrr", "ndcg5", "ndcg10", "ild", "novelty"]
list(SLICE_DEFINITIONS)

['overall', 'cold_start', 'warm', 'head', 'tail']

In [15]:
def test_slicing_partition():
    for _, group in ranking_results.groupby(["dataset", "split", "method"]):
        cold = SLICE_DEFINITIONS["cold_start"](group)
        warm = SLICE_DEFINITIONS["warm"](group)
        assert (cold | warm).all() and not (cold & warm).any()

        head = SLICE_DEFINITIONS["head"](group)
        tail = SLICE_DEFINITIONS["tail"](group)
        assert (head | tail).all() and not (head & tail).any()


test_slicing_partition()
print("ok: cold-start/warm and head/tail slices are complete, non-overlapping partitions")

ok: cold-start/warm and head/tail slices are complete, non-overlapping partitions


## Bootstrap 95% CI

Resamples impressions (SPEC.md Q4 #5), 1,000 iterations, per
`(dataset, method, split, slice, metric)`. EB-NeRD's 0-cold-start-user slice
is empty by construction (see #3) -- recorded as `None` rather than a
degenerate CI, stated plainly rather than hidden.

In [16]:
def compute_bootstrap_metrics() -> dict:
    results = {}
    for (name, split, method), group in ranking_results.groupby(["dataset", "split", "method"]):
        slices = {}
        for slice_name, mask_fn in SLICE_DEFINITIONS.items():
            sliced = group[mask_fn(group)]
            slice_metrics = {}
            for metric in METRIC_COLUMNS:
                if len(sliced) == 0:
                    slice_metrics[metric] = None
                else:
                    point, lo, hi = bootstrap_ci(
                        sliced[metric].to_numpy(), n_iterations=BOOTSTRAP_ITERATIONS, seed=BOOTSTRAP_SEED
                    )
                    slice_metrics[metric] = {"point": point, "ci_lo": lo, "ci_hi": hi}
            slices[slice_name] = slice_metrics
        results.setdefault(name, {}).setdefault(method, {})[split] = slices
    return results


bootstrap_metrics = compute_bootstrap_metrics()

In [17]:
def test_bootstrap_ci():
    n_checked = 0
    n_none = 0
    for methods in bootstrap_metrics.values():
        for splits in methods.values():
            for slices in splits.values():
                for metrics_dict in slices.values():
                    for stats in metrics_dict.values():
                        if stats is None:
                            n_none += 1
                            continue
                        assert stats["ci_lo"] <= stats["point"] <= stats["ci_hi"] + 1e-9
                        n_checked += 1
    assert n_checked > 0
    # the only expected empty slice is EB-NeRD's cold-start slice (0 users, both splits/methods)
    expected_none = len(METRIC_COLUMNS) * len(METHODS) * len(SPLITS)
    assert n_none == expected_none, f"expected {expected_none} empty (EB-NeRD cold-start) slices, got {n_none}"
    return n_checked, n_none


n_checked, n_none = test_bootstrap_ci()
print(f"ok: bootstrap CIs bracket the point estimate for {n_checked} (metric, slice) combinations; {n_none} degenerate as expected")

ok: bootstrap CIs bracket the point estimate for 216 (metric, slice) combinations; 24 degenerate as expected


## Anti-gaming (Q9) confirmation

A schema-column assertion (SPEC.md Q4 #7), not new engineering: the unified
schema never carried EB-NeRD's look-ahead fields in the first place, so
there's nothing to toggle for a with/without-unavailable-features comparison.

In [18]:
def test_no_leakage_columns():
    expected_behaviors_cols = {
        "impression_id", "dataset", "user_id", "impression_time",
        "article_ids_inview", "article_ids_clicked", "session_id", "split",
    }
    expected_history_cols = {
        "user_id", "dataset", "article_id_sequence", "timestamp_sequence",
        "read_time_sequence", "scroll_percentage_sequence",
    }
    forbidden = {"next_read_time", "next_scroll_percentage"}

    for name in DATASETS:
        behaviors_cols = set(feature_store[name]["behaviors"].columns)
        history_cols = set(feature_store[name]["history"].columns)
        assert behaviors_cols == expected_behaviors_cols
        assert history_cols == expected_history_cols
        assert forbidden.isdisjoint(behaviors_cols) and forbidden.isdisjoint(history_cols)


test_no_leakage_columns()
print(
    "ok: anti-gaming confirmed -- unified schema carries no look-ahead fields "
    "(next_read_time/next_scroll_percentage); nothing to toggle for a "
    "with/without-unavailable-features comparison"
)

ok: anti-gaming confirmed -- unified schema carries no look-ahead fields (next_read_time/next_scroll_percentage); nothing to toggle for a with/without-unavailable-features comparison


## Persist eval_metrics.json

`{method: {split: {slice: {metric: {point, ci_lo, ci_hi}}}}}` per dataset
(SPEC.md Q4 #8), plus coverage (candidate-generation-framed, not sliced) and
the anti-gaming confirmation.

In [19]:
def write_eval_metrics(dataset: str) -> Path:
    out_dir = DATA_DIR / dataset
    payload = {
        "schema_version": 1,
        "build_timestamp": datetime.now(timezone.utc).isoformat(),
        "hyperparameters": {
            "recent_n_clicks": RECENT_N_CLICKS,
            "ndcg_k_values": NDCG_K_VALUES,
            "top_list_k": TOP_LIST_K,
            "head_fraction": HEAD_FRACTION,
            "bootstrap_iterations": BOOTSTRAP_ITERATIONS,
        },
        "ranking_metrics": bootstrap_metrics[dataset],
        "coverage": {method: coverage_metrics[(dataset, method)] for method in METHODS},
        "anti_gaming_confirmed": True,
    }
    (out_dir / "eval_metrics.json").write_text(json.dumps(payload, indent=2))
    return out_dir


eval_out_dirs = {name: write_eval_metrics(name) for name in DATASETS}
eval_out_dirs

{'ebnerd': WindowsPath('C:/Users/HP/cs4406m26-assignment1c1/data/processed/ebnerd'),
 'mind': WindowsPath('C:/Users/HP/cs4406m26-assignment1c1/data/processed/mind')}

In [20]:
def test_eval_metrics_roundtrip():
    for name in DATASETS:
        path = eval_out_dirs[name] / "eval_metrics.json"
        assert path.exists()
        reloaded = json.loads(path.read_text())

        assert set(reloaded["ranking_metrics"]) == set(METHODS)
        for method in METHODS:
            assert set(reloaded["ranking_metrics"][method]) == set(SPLITS)
            for split in SPLITS:
                assert set(reloaded["ranking_metrics"][method][split]) == set(SLICE_DEFINITIONS)

        assert reloaded["coverage"]["bm25"] == coverage_metrics[(name, "bm25")]
        assert reloaded["coverage"]["embedding"] == coverage_metrics[(name, "embedding")]
        assert reloaded["anti_gaming_confirmed"] is True


test_eval_metrics_roundtrip()
print("ok: eval_metrics.json persisted and round-trips for both datasets")

ok: eval_metrics.json persisted and round-trips for both datasets


# Manual Verification Complete